# NB_04_Semantic_Model
Builds the Direct Lake semantic model **as code** with Semantic Link Labs: tables, relationships,
date table and measures. Safe to re-run; existing relationships and measures are skipped.

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
dataset = "SM_ControlTower"
lakehouse = "LH_ControlTower"
schema = "dbo"

In [ ]:
import inspect
import sempy.fabric as fabric
from sempy_labs.directlake import generate_direct_lake_semantic_model
from sempy_labs.tom import connect_semantic_model

TABLES = ["dim_date", "dim_customer", "dim_product", "dim_supplier",
          "fact_sales", "fact_shipment", "fact_invoice", "fact_payment", "silver_dq_rejects"]

## 1. Create the model from the gold tables

In [ ]:
ds = fabric.list_datasets()
existing = ds["Dataset Name"].tolist() if "Dataset Name" in ds.columns else []
if dataset in existing:
    print(f"{dataset} already exists, skipping creation")
else:
    # the function's arguments were renamed between library versions, so pass whichever it accepts
    accepted = inspect.signature(generate_direct_lake_semantic_model).parameters
    args = {"dataset": dataset}
    if "schema" in accepted:                      # older versions take the schema separately
        args["lakehouse_tables"], args["lakehouse"], args["schema"] = TABLES, lakehouse, schema
    else:                                         # newer versions expect schema-qualified table names
        args["tables"], args["source"] = [f"{schema}.{t}" for t in TABLES], lakehouse
    generate_direct_lake_semantic_model(**args)

## 2. Relationships, date table and measures

In [ ]:
RELATIONSHIPS = [  # (from_table, from_column, to_table, to_column, is_active)
    ("fact_sales",    "invoice_date", "dim_date",     "date",        True),
    ("fact_sales",    "customer_id",  "dim_customer", "customer_id", True),
    ("fact_sales",    "sku",          "dim_product",  "sku",         True),
    ("fact_invoice",  "invoice_date", "dim_date",     "date",        True),
    ("fact_invoice",  "customer_id",  "dim_customer", "customer_id", True),
    ("fact_payment",  "txn_date",     "dim_date",     "date",        True),
    ("fact_payment",  "customer_id",  "dim_customer", "customer_id", True),
    ("fact_shipment", "po_date",      "dim_date",     "date",        True),
    ("fact_shipment", "arrival_date", "dim_date",     "date",        False),   # USERELATIONSHIP
    ("fact_shipment", "supplier_id",  "dim_supplier", "supplier_id", True),
    ("fact_shipment", "sku",          "dim_product",  "sku",         True),
]

GBP, PCT, NUM, DAYS = "£#,0", "0.0%", "#,0", "#,0.0"
ARRIVED = 'fact_shipment[status] = "Arrived"'

MEASURES = [  # (table, name, expression, format, display folder)
    ("fact_sales", "Net Sales", "SUM ( fact_sales[net_amount_gbp] )", GBP, "Sales"),
    ("fact_sales", "Landed COGS", "SUM ( fact_sales[landed_cost_gbp] )", GBP, "Sales"),
    ("fact_sales", "Gross Margin", "[Net Sales] - [Landed COGS]", GBP, "Sales"),
    ("fact_sales", "Gross Margin %", "DIVIDE ( [Gross Margin], [Net Sales] )", PCT, "Sales"),
    ("fact_sales", "Net Sales PY", "CALCULATE ( [Net Sales], SAMEPERIODLASTYEAR ( dim_date[date] ) )", GBP, "Sales"),
    ("fact_sales", "Net Sales YoY %", "DIVIDE ( [Net Sales] - [Net Sales PY], [Net Sales PY] )", PCT, "Sales"),

    ("fact_shipment", "Shipments Arrived", f"CALCULATE ( COUNTROWS ( fact_shipment ), {ARRIVED} )", NUM, "Supplier"),
    ("fact_shipment", "OTIF %",
     "DIVIDE ( CALCULATE ( COUNTROWS ( fact_shipment ), fact_shipment[is_otif] = TRUE () ), [Shipments Arrived] )", PCT, "Supplier"),
    ("fact_shipment", "On-Time %",
     "DIVIDE ( CALCULATE ( COUNTROWS ( fact_shipment ), fact_shipment[is_on_time] = TRUE () ), [Shipments Arrived] )", PCT, "Supplier"),
    ("fact_shipment", "Fill Rate %",
     f"CALCULATE ( DIVIDE ( SUM ( fact_shipment[qty_received] ), SUM ( fact_shipment[qty_ordered] ) ), {ARRIVED} )", PCT, "Supplier"),
    ("fact_shipment", "Avg Delay Days", f"CALCULATE ( AVERAGE ( fact_shipment[delay_days] ), {ARRIVED} )", DAYS, "Supplier"),
    ("fact_shipment", "Avg Lead Time Days", f"CALCULATE ( AVERAGE ( fact_shipment[lead_time_days] ), {ARRIVED} )", DAYS, "Supplier"),
    ("fact_shipment", "In Transit Value",
     'CALCULATE ( SUMX ( fact_shipment, fact_shipment[qty_ordered] * fact_shipment[unit_cost] * fact_shipment[fx_rate_to_gbp] ), '
     'fact_shipment[status] = "In Transit" )', GBP, "Supplier"),

    ("fact_shipment", "Goods Value", "SUM ( fact_shipment[goods_value_gbp] )", GBP, "Landed cost"),
    ("fact_shipment", "Freight + Duty",
     "SUM ( fact_shipment[freight_cost_gbp] ) + SUM ( fact_shipment[duty_gbp] )", GBP, "Landed cost"),
    ("fact_shipment", "Landed Cost Uplift %",
     f"CALCULATE ( DIVIDE ( [Freight + Duty], [Goods Value] ), {ARRIVED} )", PCT, "Landed cost"),
    ("fact_shipment", "Landed Cost Uplift % (by arrival)",
     "CALCULATE ( [Landed Cost Uplift %], USERELATIONSHIP ( fact_shipment[arrival_date], dim_date[date] ) )", PCT, "Landed cost"),

    ("fact_invoice", "Invoiced", "SUM ( fact_invoice[invoice_amount_gbp] )", GBP, "Receivables"),
    ("fact_invoice", "Outstanding AR", "SUM ( fact_invoice[outstanding_gbp] )", GBP, "Receivables"),
    ("fact_invoice", "Overdue AR", "CALCULATE ( [Outstanding AR], fact_invoice[days_overdue] > 0 )", GBP, "Receivables"),
    ("fact_invoice", "Overdue % of AR", "DIVIDE ( [Overdue AR], [Outstanding AR] )", PCT, "Receivables"),
    ("fact_invoice", "Avg Days to Pay", "AVERAGE ( fact_invoice[days_to_pay] )", DAYS, "Receivables"),
    ("fact_invoice", "Weighted Days to Pay",
     "DIVIDE ( SUMX ( fact_invoice, fact_invoice[days_to_pay] * fact_invoice[paid_amount_gbp] ), "
     "CALCULATE ( SUM ( fact_invoice[paid_amount_gbp] ), NOT ISBLANK ( fact_invoice[days_to_pay] ) ) )", DAYS, "Receivables"),
    ("fact_invoice", "Short-Paid Invoices",
     'CALCULATE ( COUNTROWS ( fact_invoice ), fact_invoice[status] = "Short-paid" )', NUM, "Receivables"),

    ("fact_payment", "Cash Received", "SUM ( fact_payment[amount_gbp] )", GBP, "Reconciliation"),
    ("fact_payment", "Unmatched Cash", "CALCULATE ( [Cash Received], fact_payment[is_matched] = FALSE () )", GBP, "Reconciliation"),
    ("fact_payment", "Bank Lines", "COUNTROWS ( fact_payment )", NUM, "Reconciliation"),
    ("fact_payment", "Auto-Match Rate %",
     "DIVIDE ( CALCULATE ( COUNTROWS ( fact_payment ), fact_payment[is_matched] = TRUE () ), COUNTROWS ( fact_payment ) )", PCT, "Reconciliation"),

    ("silver_dq_rejects", "Rejected Rows", "COUNTROWS ( silver_dq_rejects )", NUM, "Data quality"),
]

with connect_semantic_model(dataset=dataset, readonly=False) as tom:
    have_rels = {(r.FromTable.Name, r.FromColumn.Name, r.ToTable.Name, r.ToColumn.Name) for r in tom.model.Relationships}
    for ft, fc, tt, tc, active in RELATIONSHIPS:
        if (ft, fc, tt, tc) not in have_rels:
            tom.add_relationship(from_table=ft, from_column=fc, to_table=tt, to_column=tc,
                                 from_cardinality="Many", to_cardinality="One", is_active=active)

    tom.mark_as_date_table(table_name="dim_date", column_name="date")

    have_measures = {m.Name for m in tom.all_measures()}
    for table, name, expr, fmt, folder in MEASURES:
        if name not in have_measures:
            tom.add_measure(table_name=table, measure_name=name, expression=expr,
                            format_string=fmt, display_folder=folder)

    print(f"{len(list(tom.model.Relationships))} relationships, {len(list(tom.all_measures()))} measures")

## 3. Recalculate
New relationships hold no data until the model is recalculated.

In [ ]:
import sempy_labs as labs
labs.refresh_semantic_model(dataset=dataset, refresh_type="calculate")

## 4. Smoke test: query the model with DAX

In [ ]:
fabric.evaluate_dax(dataset, """
EVALUATE ROW (
    "Net Sales", [Net Sales],
    "Gross Margin %", [Gross Margin %],
    "OTIF %", [OTIF %],
    "Landed Cost Uplift %", [Landed Cost Uplift %],
    "Overdue AR", [Overdue AR],
    "Auto-Match Rate %", [Auto-Match Rate %],
    "Rejected Rows", [Rejected Rows]
)
""")

In [ ]:
fabric.evaluate_dax(dataset, """
EVALUATE
SUMMARIZECOLUMNS ( dim_supplier[supplier_name], "OTIF %", [OTIF %], "Avg Delay Days", [Avg Delay Days],
                   "Landed Cost Uplift %", [Landed Cost Uplift %] )
ORDER BY [OTIF %]
""")